# 05 - Final fit and validation premiums

The selected setup (notebook 04) is fitted on **all training months** (February to July) and prices the 4,155
validation orders (31 July to 14 October). The base-price rebuild may also learn from the base prices that the
validation file gives: they are inputs, not outcomes. The validation closing times are not used.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [2]:
from dataclasses import fields
from pricing.data import load_orders, load_btc, VALID_FILE
from pricing.features import market_features
from pricing.pipeline import HybridPricer, Config

sel = json.loads((OUT / "selected_config.json").read_text())
cfg = Config(**{k: tuple(v) if isinstance(v, list) else v for k, v in sel["config"].items()})
print("selected setup:", sel["name"]); print(cfg)
train, valid = load_orders("train"), load_orders("valid")
btc = load_btc(); mkt = market_features(btc)
Xtr = pd.read_parquet(OUT / "features_train.parquet").join(pd.read_parquet(OUT / "external_train.parquet"))
Xva = pd.read_parquet(OUT / "features_valid.parquet").join(pd.read_parquet(OUT / "external_valid.parquet"))
fit_rows = train[train["has_outcome"]]
cutoff = valid["created"].min().floor("min")
pricer = HybridPricer(cfg).fit(fit_rows, Xtr.loc[fit_rows.index], btc, mkt, cutoff=cutoff, extra_base=(valid, Xva))
out = pricer.predict(valid, Xva)
print("calibration constant:", round(pricer.risk.const, 3))
pricer.risk.factors().round(3)

selected setup: hybrid_events
Config(k=3.0, cap_lo=0.85, cap_hi=1.6, risk_features=('event_in_window', 'hours_to_event', 'hours_since_event'), risk_kind='glm', risk_alpha=0.1, size_cap=(0.8, 1.5), ref_kind='model')


calibration constant: 1.361


event_in_window     1.227
hours_to_event      0.999
hours_since_event   1.000
dtype: float64

## Output file: both a premium and a multiplier for every order

- `premiumAmount_USDC`: the premium for the order (premium per BTC x size).
- `pricingMultiplier`: premium per BTC / anchor. Where Jasper's base price exists, the anchor **is** that base price, so
  final premium = base premium x multiplier. Where it is missing, the anchor is our rebuilt base price (`anchor_source` = "rebuilt").

In [3]:
raw = pd.read_csv(VALID_FILE)
sub = raw.copy()
sub["premiumAmount_USDC"] = out["premiumAmount_USDC"].to_numpy()
sub["pricingMultiplier"] = out["pricing_multiplier"].to_numpy()
sub["premium_per_BTC_USDC"] = out["premium_per_BTC"].to_numpy()
sub["anchor_source"] = out["anchor_source"].to_numpy()
sub["anchor_per_BTC_USDC"] = out["anchor_per_BTC"].to_numpy()
sub["expected_payout_per_BTC_USDC"] = out["expected_payout_per_BTC"].to_numpy()
sub["reference_price_per_BTC_USDC"] = out["reference_price_per_BTC"].to_numpy()
sub.to_csv(OUT / "validation_premiums.csv", index=False)
print(f"written: outputs/validation_premiums.csv  ({len(sub):,} rows)")
sub.head()

written: outputs/validation_premiums.csv  (4,155 rows)


,premiumAmount_USDC,createdAt,closedAt,expirationDate,optionType,productType,quantity_BTC,strikePrice,realizedProfit_USDC,optionType_label,orderPrice_per_BTC_USDC,apiBasePrice_per_BTC_USDC,markup_per_BTC_USDC,pricingMultiplier,premium_per_BTC_USDC,anchor_source,anchor_per_BTC_USDC,expected_payout_per_BTC_USDC,reference_price_per_BTC_USDC
0,0.302,2026/7/31 17:11,2026-07-31 18:11:29,2026/7/31 18:11,1,3600,0.002,"62,786.950",NaN,Put,NaN,67.940,NaN,1.895,128.716,jasper,67.940,85.703,129.039
1,0.401,2026/7/31 17:14,2026-07-31 17:44:57,2026/7/31 17:44,0,1800,0.004,"62,819.750",NaN,Call,NaN,48.047,NaN,1.909,91.726,jasper,48.047,60.611,93.345
2,0.171,2026/7/31 17:14,2026-07-31 17:44:38,2026/7/31 17:44,0,1800,0.002,"62,813.750",NaN,Call,NaN,48.047,NaN,1.936,93.019,jasper,48.047,60.611,97.225
3,0.330,2026/7/31 17:15,2026-07-31 17:45:08,2026/7/31 17:44,0,1800,0.004,"62,818.050",NaN,Call,NaN,48.035,NaN,1.930,92.703,jasper,48.035,60.597,96.317
4,0.455,2026/7/31 17:23,2026-07-31 18:22:59,2026/7/31 18:22,0,3600,0.004,"62,788.250",NaN,Call,NaN,68.000,NaN,1.888,128.411,jasper,68.000,85.791,127.862


## Checks

In [4]:
assert len(sub) == len(raw) == 4155
assert (sub["createdAt"] == raw["createdAt"]).all()          # same order as the input file
assert sub["premiumAmount_USDC"].notna().all() and (sub["premiumAmount_USDC"] > 0).all()
ratio = out["premium_per_BTC"] / out["reference_price_per_BTC"]
floor = out["premium_per_BTC"] <= out["expected_payout_per_BTC"] + 1e-9
assert ((ratio >= cfg.cap_lo - 1e-9) & ((ratio <= cfg.cap_hi + 1e-9) | floor)).all()
print("all checks passed")
print(f"orders priced at the expected-payout floor (above the upper cap): {int((ratio > cfg.cap_hi + 1e-9).sum())}")

all checks passed
orders priced at the expected-payout floor (above the upper cap): 0


## How reliable is the rebuilt base price?

September and October have no Jasper base price at all. To estimate the error there, we refit the rebuild
**without** any validation base prices and compare it with the real base prices of the August orders.

In [5]:
check = HybridPricer(cfg).fit(fit_rows, Xtr.loc[fit_rows.index], btc, mkt, cutoff=cutoff)
o2 = check.predict(valid, Xva)
has = valid["base"].notna()
err = o2.loc[has, "base_rebuilt_per_BTC"] / valid.loc[has, "base"] - 1
rebuild_err = {"median_abs_error": float(err.abs().median()), "p90_abs_error": float(err.abs().quantile(0.9)),
               "mean_error": float(err.mean())}
print({k: round(v, 3) for k, v in rebuild_err.items()})

{'median_abs_error': 0.091, 'p90_abs_error': 0.217, 'mean_error': 0.091}


## Our premium against Jasper's real price (544 validation orders where the file gives it)

In [6]:
known = valid["paid"].notna()
r_known = out.loc[known, "premium_per_BTC"] / valid.loc[known, "paid"]
print(f"orders: {known.sum()}  median ratio: {r_known.median():.2f}  lower than Jasper: {np.mean(r_known < 1):.0%}  above 1.5x: {np.mean(r_known > 1.5):.0%}")

orders: 544  median ratio: 1.19  lower than Jasper: 30%  above 1.5x: 4%


In [7]:
v = valid.join(out)
v["month"] = v["created"].dt.to_period("M").astype(str)
by_m = v.groupby("month").agg(orders=("qty", "size"), rebuilt_anchor_share=("anchor_source", lambda s: (s == "rebuilt").mean()),
                              median_premium_per_BTC=("premium_per_BTC", "median"),
                              median_anchor_per_BTC=("anchor_per_BTC", "median"),
                              premium_total_usd=("premiumAmount_USDC", "sum"),
                              expected_payout_usd=("expected_payout_per_BTC", lambda s: (s * v.loc[s.index, "qty"]).sum()))
by_m.to_csv(OUT / "validation_by_month.csv")
summary = {"orders": int(len(sub)), "selected": sel["name"], "rebuild_error_august": rebuild_err,
           "known_price_rows": int(known.sum()), "known_price_median_ratio": float(r_known.median()),
           "known_price_share_lower": float(np.mean(r_known < 1)), "known_price_share_above_1.5": float(np.mean(r_known > 1.5)),
           "premium_total_usd": float(sub["premiumAmount_USDC"].sum()),
           "expected_payout_total_usd": float((out["expected_payout_per_BTC"] * valid["qty"]).sum()),
           "calibration_constant": pricer.risk.const,
           "risk_factors": {k: float(x) for k, x in pricer.risk.factors().items()},
           "k": cfg.k, "cap_lo": cfg.cap_lo, "cap_hi": cfg.cap_hi}
(OUT / "validation_summary.json").write_text(json.dumps(summary, indent=1))
by_m.round(3)

,orders,rebuilt_anchor_share,median_premium_per_BTC,median_anchor_per_BTC,premium_total_usd,expected_payout_usd
month,,,,,,
2026-07,118,0.000,93.937,48.199,70.449,46.676
2026-08,2129,0.135,189.010,99.209,"98,143.863","73,827.793"
2026-09,1340,1.000,225.373,118.621,"87,631.449","64,241.707"
2026-10,568,1.000,213.120,109.987,"30,917.894","22,134.503"
